<div style="text-align:center; margin-bottom:30px;">

<img src="figures/ucirvine_logo.png" width="220" style="margin-bottom:20px;">

<h2 style="margin-bottom:20px; font-weight:normal;">SkillCraft1: Multinomial Skill Classification Using Machine Learning</h2>

<div style="margin-bottom:5px; font-weight:bold;">William Chen</div>
<div style="margin-bottom:5px;">University of California, Irvine</div>
<div style="margin-bottom:5px;">Instructor: Prof. Sam Behseta</div>
<div style="margin-bottom:0px;">DATA 200BP: Intermediate Probability and Statistical Theory II</div>

</div>

# Abstract

This project extends the work of Thompson et al. (2013), who used pairwise binary
classification with Conditional Inference Forest to study variable importance across
StarCraft II skill levels. While their approach provided insights into how predictors
of expertise change across leagues, it did not address the full complexity of multi-class
skill prediction or the class imbalance inherent in the SkillCraft1 dataset. This study
reformulates the problem as a 6-class multinomial classification task, predicting player
league rankings from Bronze (1) to Masters (6).

Three machine learning models are evaluated — Multinomial Logistic Regression,
Decision Tree, and Random Forest — each trained under three class imbalance strategies:
Baseline, Class Weights, and SMOTE combined with Tomek Links. Model performance is
assessed using overall accuracy and minority class recall on a held-out test set.

Results show that Random Forest with Class Weights achieves the best overall balance,
with an accuracy of 0.4070 and a meaningful improvement in minority class recall
compared to the baseline. In a direct pairwise comparison replicating Thompson et al.'s
setup, our best model outperforms their Conditional Inference Forest results in 3 out
of 4 league pairs. These findings suggest that class weighting is an effective and
practical strategy for handling imbalance in multi-class skill classification tasks.

# 1. Introduction

## 1.1 Nature of the Project

Cognitive science has long studied expertise by comparing experts with novices. Thompson et al. (2013) challenged the assumptions underlying such comparisons by analyzing StarCraft II telemetry data from 3,395 players across competitive leagues. They discovered that variable importance is not static — APM dominates at lower skill levels, but at higher levels, attentional efficiency metrics become more predictive.

Their method used pairwise binary classifiers built with Conditional Inference Forests, each comparing players two leagues apart. This design was deliberate: neighboring leagues are not cleanly separable because league placement does not perfectly reflect objective skill. They found that classifiers performed poorly on adjacent leagues and had significantly more success when the distance between classes was at least two.

StarCraft II is well-suited for this type of study. Unlike chess, an average RTS game contains ~1,635 moves, providing dense behavioral data in a naturalistic setting. The SkillCraft1 dataset contains 20 behavioral and demographic variables, making it the largest expertise study of its kind.

## 1.2 Aims

This project extends Thompson et al.'s pairwise approach in two directions. First, we
replicate their pairwise binary classifiers using our best-performing configuration to
benchmark our methodology against their Conditional Inference Forest results. Second,
we develop a direct 6-class multinomial prediction model that simultaneously classifies
players across all skill levels — a more challenging problem that requires handling
severe class imbalance and non-linear decision boundaries.

To address class imbalance, three strategies are compared: baseline classification,
class weight adjustment, and SMOTE combined with Tomek Links. This allows us to
investigate whether imbalance-aware training improves prediction of minority classes.

## 1.3 Goals

The primary goal of this project is to develop an effective 6-class multinomial
classifier for Leagues 1 through 6 that balances overall accuracy with minority class
recall. To achieve this, we compare three machine learning models — Multinomial Logistic
Regression, Decision Tree, and Random Forest — each trained under three imbalance
strategies: Baseline, Class Weights, and SMOTE combined with Tomek Links. Additionally,
we replicate Thompson et al.'s pairwise binary classification setup to benchmark our
best-performing model against their Conditional Inference Forest results.

A secondary goal is to examine whether demographic features such as Age and HoursPerWeek
contribute meaningfully to skill prediction beyond in-game behavioral metrics.

# 2. Literature Review

## 2.1 Skill Classification in Competitive Gaming

Skill assessment in competitive games bridges expertise research with machine learning. 
Unlike laboratory settings, games provide naturalistic data where players compete under 
identical conditions. Real-time strategy games like StarCraft II are particularly rich: 
they demand simultaneous management of multiple objectives and high-frequency decision-making, 
making them ideal for studying expertise across a continuous skill spectrum.

## 2.2 Thompson et al. (2013): SkillCraft1 Baseline

Thompson et al. (2013) introduced the SkillCraft1 dataset, comprising 3,395 players
across 7 leagues with 20 behavioral and demographic variables. Their primary objective
was to examine how variable importance changes across skill levels, rather than to
maximize classification accuracy. Using pairwise binary classifiers built with
Conditional Inference Forests (ntree = 1000, mtry = 5), they compared leagues two
levels apart to avoid the separability issues inherent in adjacent league comparisons.

Their analysis revealed that adjacent leagues are difficult to separate because league
placement does not perfectly reflect objective skill. They also found that variable
importance shifts across the skill spectrum — APM predicts skill at lower levels, while
attentional efficiency metrics such as Action Latency and Gap Between PACs become
dominant at higher levels. Grandmaster players were excluded due to their small sample
size (35 players), and 55 professional players were added from publicly available
tournament replays to form the highest skill group.

This project extends their work by replicating the pairwise comparison using RF +
Class Weights — a standard Random Forest with class weighting, as opposed to their
Conditional Inference Forest — and by attempting direct 6-class multinomial prediction,
a harder problem that Thompson et al. did not pursue due to the separability limitations
they observed.

## 2.3 Class Imbalance Handling Methods

Class imbalance occurs when certain classes are significantly smaller than others,
causing models to favor majority classes and underperform on minority ones. Three
common strategies for addressing this issue are evaluated in this study.

SMOTE generates synthetic minority samples by interpolating between existing
observations and their nearest neighbors. This increases representation without
duplication, but may produce samples that do not reflect realistic data patterns.

Tomek Links remove majority class samples that are nearest neighbors of minority
class samples, cleaning the decision boundary. This is often combined with SMOTE
to provide both augmentation and boundary clarity.

Class Weighting adjusts the loss function to penalize misclassification of minority
samples more heavily, typically using inverse frequency weighting. Unlike resampling
methods, it preserves the original data distribution entirely.

## 2.4 Machine Learning Models for Classification

Multinomial Logistic Regression produces linear decision boundaries and interpretable
coefficients, but may underperform when feature-class relationships are nonlinear.

Decision Trees partition the feature space using binary splits, naturally handling
nonlinearity. However, they are prone to overfitting and require pruning to generalize.

Random Forests aggregate many decision trees through bagging and random feature
selection, reducing overfitting and improving robustness at the cost of interpretability.

These three models are compared across three imbalance strategies to identify the
most effective combination for 6-class skill classification.

# 3. Data Introduction

## 3.1 SkillCraft1 Dataset Overview

The SkillCraft1 dataset contains 3,395 StarCraft II player observations across 8
leagues. After removing League 8 (systematic missing data in all demographic features)
and League 7 (Grandmaster, excluded per Thompson et al. due to small sample size),
the analysis uses 3,340 observations across 6 leagues: Bronze, Silver, Gold, Platinum,
Diamond, and Masters.

The dataset is severely class-imbalanced, with Bronze players comprising only about
5% of observations while Platinum and Diamond together account for nearly half.

## 3.2 Data Features & Characteristics

The dataset contains 20 variables: 3 demographic (Age, HoursPerWeek, TotalHours) and
17 behavioral, capturing in-game actions such as APM, hotkey usage, map control,
decision-making efficiency, and unit production, all normalized per minute of gameplay.

# 4. Exploratory Data Analysis (EDA)

## 4.1 Data Loading and Structure

In [1]:
library(ggplot2)
library(rpart)      
library(rpart.plot)
library(cowplot)
library(caret)
library(randomForest)
library(tidyr)
library(GGally)
library(corrplot)
library(pROC)
library(nnet)
library(car)
library(dplyr)
library(smotefamily)
library(themis)
library(nnet)
options(warn = -1)

Loading required package: lattice

randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.


Attaching package: ‘randomForest’


The following object is masked from ‘package:ggplot2’:

    margin


corrplot 0.95 loaded

Type 'citation("pROC")' for a citation.


Attaching package: ‘pROC’


The following objects are masked from ‘package:stats’:

    cov, smooth, var


Loading required package: carData


Attaching package: ‘dplyr’


The following object is masked from ‘package:car’:

    recode


The following object is masked from ‘package:randomForest’:

    combine


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: recipes


Attaching package: ‘recipes’


The following object is masked from ‘package:stats’:

    step




In [ ]:
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00272/SkillCraft1_Dataset.csv"
skillcraft.data <- read.csv(URL)

In [ ]:
str(skillcraft.data)

The dataset contains 3,395 observations and 20 variables. Based on the structure, the following conversions are required:

- LeagueIndex: ranges from 1 (Bronze) to 8 (Professional), converted to factor for classification
- GameID: serves only as an identifier with no predictive value, removed
- Age, HoursPerWeek, TotalHours: stored as character type, converted to numeric

## 4.2 Data Type Conversion

In [ ]:
skillcraft.data$LeagueIndex <- as.factor(skillcraft.data$LeagueIndex)
skillcraft.data$GameID <- NULL
skillcraft.data$Age <- as.numeric(skillcraft.data$Age)
skillcraft.data$HoursPerWeek <- as.numeric(skillcraft.data$HoursPerWeek)
skillcraft.data$TotalHours <- as.numeric(skillcraft.data$TotalHours)

LeagueIndex is converted to factor for classification. GameID is removed as it has
no predictive value. Age, HoursPerWeek, and TotalHours are converted from character
to numeric, which introduces NA values for non-numeric entries such as "?".

## 4.3 Summary Statistics

In [ ]:
summary(skillcraft.data)

From the summary, Age, HoursPerWeek, and TotalHours contain missing values (NA).
Additionally, TotalHours shows an abnormally large maximum value of 1,000,000,
suggesting the presence of an outlier that will need to be addressed in preprocessing.

## 4.4 Class Distribution

In [ ]:
ggplot(data = skillcraft.data, mapping = aes(x = LeagueIndex)) +
  geom_bar(fill = "steelblue") +
  geom_text(stat = "count", aes(label = ..count..), vjust = -0.5) +
  scale_x_discrete(labels = c("1" = "Bronze", "2" = "Silver", "3" = "Gold",
                               "4" = "Platinum", "5" = "Diamond", "6" = "Master",
                               "7" = "GM", "8" = "Pro")) +
  labs(title = "Player Distribution by League",
       x = "League", y = "Count")

The distribution of players across leagues is highly imbalanced. Platinum (League 4)
and Diamond (League 5) account for the majority of observations, while Grandmaster
(League 7) and Professional (League 8) have very few samples. This imbalance will be
addressed in Section 5 and Section 6.

## 4.5 Feature Distribution by League

We examine four features that may not intuitively correlate with league rank.

In [ ]:
features <- c("Age", "TotalHours", "WorkersMade", "MinimapAttacks")

df_long <- pivot_longer(skillcraft.data[, c("LeagueIndex", features)],
                        cols = -LeagueIndex,
                        names_to = "feature",
                        values_to = "value")

ggplot(df_long, aes(x = LeagueIndex, y = value, fill = LeagueIndex)) +
  geom_boxplot(outlier.size = 0.5, alpha = 0.7) +
  facet_wrap(~ feature, scales = "free_y", ncol = 2) +
  labs(title = "Feature Distribution by League",
       x = "League Index", y = "Value") +
  theme(legend.position = "none")

Based on the boxplots:

- Age: Median age is stable around 20–22 across all leagues, suggesting age alone is not a strong indicator of skill level
- MinimapAttacks: Boxes are narrow and close to zero across most leagues, though higher leagues show slightly wider distributions
- TotalHours: An extreme outlier (1,000,000 hours) dominates the y-axis, confirming the need for outlier capping in preprocessing
- WorkersMade: League 6 shows the highest median rather than League 7 or 8, suggesting worker production does not linearly increase with rank

## 4.6 Outlier Analysis (TotalHours)

In [ ]:
percentiles <- quantile(skillcraft.data$TotalHours, 
                        probs = c(0.90, 0.95, 0.99, 0.999, 1.0), 
                        na.rm = TRUE)
print(percentiles)

In [ ]:
cap_99 <- quantile(skillcraft.data$TotalHours, probs = 0.99, na.rm = TRUE)

ggplot(skillcraft.data[skillcraft.data$TotalHours <= cap_99, ], 
       aes(x = TotalHours)) +
  geom_histogram(bins = 50, fill = "steelblue") +
  labs(title = "TotalHours Distribution (Capped at 99th Percentile)", 
       x = "TotalHours", y = "Count")

TotalHours contains an extreme outlier of 1,000,000 hours. After capping at the 99th
percentile (2,520 hours), the distribution shows a right-skewed pattern with most
players under 1,000 hours. Actual outlier handling is applied in Section 5.

## 4.7 Missing Value Analysis

We investigate whether missing values are concentrated in specific leagues or
spread randomly, as this determines the appropriate treatment.

In [ ]:
show_missing <- function(league_id) {
  rows <- skillcraft.data[skillcraft.data$LeagueIndex == league_id, ]
  counts <- colSums(is.na(rows))
  counts <- counts[counts > 0]
  if (length(counts) == 0) return(NULL)
  data.frame(
    League       = paste0("League ", league_id),
    Feature      = names(counts),
    Missing      = as.integer(counts),
    Total        = nrow(rows),
    Missing_Rate = paste0(round(counts / nrow(rows) * 100, 1), "%")
  )
}

result <- as.data.frame(rbind(show_missing(5), show_missing(8)))
print(result, row.names = FALSE)

All 55 League 8 observations are missing Age, HoursPerWeek, and TotalHours (100%),
indicating systematic missingness — likely because professional players' data was
collected from tournament records rather than standard ladder profiles. In contrast,
League 5 has only 2–3 missing entries (< 0.3%), consistent with random gaps.
This supports excluding League 8 entirely rather than attempting imputation.

## 4.8 Feature Correlations

In [ ]:
numeric_features <- c("LeagueIndex", "APM", "ActionLatency", 
                      "UniqueHotkeys", "AssignToHotkeys")

corrplot(cor(skillcraft.data[, numeric_features] %>% 
               mutate(LeagueIndex = as.numeric(LeagueIndex)), 
             use = "complete.obs"),
         method = "color",
         type = "upper",
         tl.cex = 0.8,
         addCoef.col = "black",
         number.cex = 0.6)

The heatmap reveals several notable correlations with LeagueIndex:

- APM (0.66) and AssignToHotkeys (0.53) show strong positive correlations, indicating higher-ranked players have faster actions and more hotkey usage
- ActionLatency (-0.67) shows the strongest negative correlation, confirming that lower reaction time is associated with higher league rank
- APM and ActionLatency are strongly negatively correlated (-0.72), suggesting potential multicollinearity to be addressed in modeling

# 5. Data Preprocessing

## 5.1 Exclusion of Leagues 7 & 8

In [ ]:
skillcraft.data <- skillcraft.data[skillcraft.data$LeagueIndex != 8, ]
skillcraft.data <- skillcraft.data[skillcraft.data$LeagueIndex != 7, ]
skillcraft.data$LeagueIndex <- droplevels(skillcraft.data$LeagueIndex)

League 8 (Professional) is excluded due to 100% systematic missing values in all
demographic features. League 7 (Grandmaster) is excluded per Thompson et al. (2013)
due to its small sample size and analytic difficulties. The remaining dataset contains
3,340 observations across 6 leagues (Bronze through Masters).

## 5.2 Handle Outliers in TotalHours

In [ ]:
cap_99 <- quantile(skillcraft.data$TotalHours, probs = 0.99, na.rm = TRUE)
skillcraft.data$TotalHours <- pmin(skillcraft.data$TotalHours, cap_99)

TotalHours is capped at the 99th percentile to remove extreme outliers while
preserving the overall distribution.

## 5.3 Handle Remaining Missing Values

In [ ]:
set.seed(42)
skillcraft.data.imputed <- rfImpute(LeagueIndex ~ ., data = skillcraft.data, iter = 6)

The remaining missing values (5 observations in League 5) are imputed using rfImpute,
which uses Random Forest proximity to estimate missing values iteratively over 6
iterations. The OOB error stabilizes around 58–59%, indicating convergence.

## 5.4 Train/Test Split

In [ ]:
set.seed(42)
train.index <- sample(1:nrow(skillcraft.data.imputed), 0.8 * nrow(skillcraft.data.imputed))
train.data <- skillcraft.data.imputed[train.index, ]
test.data  <- skillcraft.data.imputed[-train.index, ]

The data is split into 80% training (n=2,644) and 20% test (n=661) sets with a fixed random seed.

# 6. Methods

## 6.1 Feature Selection & Tuning

### Multicollinearity Check (VIF)

In [ ]:
lm.proxy <- lm(as.numeric(LeagueIndex) ~ ., data = train.data)
vif.result <- vif(lm.proxy)

vif.df <- data.frame(
  Feature = names(vif.result),
  VIF     = round(vif.result, 3)
)
print(vif.df, row.names = FALSE)

VIF analysis identifies four features with high multicollinearity: APM (41.05),
SelectByHotkeys (13.76), NumberOfPACs (14.96), and ActionsInPAC (9.42). These
overlap substantially with other action-based and PAC-related features. ActionLatency
(5.24) is also moderately inflated. All remaining features have VIF below 3.

### Backward Elimination

In [ ]:
invisible(capture.output(
  step.result <- stats::step(lm.proxy, direction = "backward")
))

step.result

In [ ]:
selected.features <- names(coef(step.result))[-1]
all.features      <- colnames(train.data[, -1])
removed.features  <- all.features[!all.features %in% selected.features]

cat("Removed", length(removed.features), "features:", paste(removed.features, collapse = ", "), "\n")

Backward elimination iteratively removes features that contribute least to model fit
based on AIC. The final model retains 14 features, removing APM, MinimapRightClicks,
ComplexAbilitiesUsed, and Age.

### Decay Tuning

We tune the `decay` parameter using 5-fold cross-validation to find the optimal 
L2 regularization strength for the multinomial logistic regression model.

In [ ]:
set.seed(42)
ctrl <- trainControl(method = "cv", number = 5)

invisible(capture.output(
  cv.model <- train(LeagueIndex ~ HoursPerWeek + TotalHours +
                      SelectByHotkeys + AssignToHotkeys + UniqueHotkeys +
                      MinimapAttacks + NumberOfPACs + GapBetweenPACs +
                      ActionLatency + ActionsInPAC + TotalMapExplored +
                      WorkersMade + UniqueUnitsMade + ComplexUnitsMade,
                    data = train.data,
                    method = "multinom",
                    trControl = ctrl,
                    maxit = 300,
                    trace = FALSE)
))

cv.model

In [ ]:
cv.results <- data.frame(
  Decay    = cv.model$results$decay,
  Accuracy = round(cv.model$results$Accuracy, 4),
  Kappa    = round(cv.model$results$Kappa, 4)
)
print(cv.results, row.names = FALSE)

The results show that decay = 0 achieves the highest accuracy (0.4217), meaning no
regularization is optimal. The final model is built with decay = 0.

## 6.2 Class Imbalance Strategies

### Imbalance Analysis

As shown in Section 4.4, the dataset is heavily imbalanced. League 1 (Bronze)
comprises only 5.05% of observations, corresponding to approximately 134 training
samples after the 80/20 split. Three strategies are applied to address this imbalance,
as described below.

### Class Weights

In [ ]:
calculate_class_weights <- function(data) {
  class.counts <- table(data$LeagueIndex)
  max.count <- max(class.counts)
  class.weights <- max.count / class.counts
  sample.weights <- as.numeric(class.weights[as.character(data$LeagueIndex)])
  return(sample.weights)
}

Class weighting adjusts the loss function so that misclassifying a minority class
sample incurs a higher penalty, without modifying the training data. Weights are
calculated using inverse frequency, so smaller classes receive proportionally higher
weights.

### SMOTE + Tomek

In [ ]:
set.seed(42)

train.smote <- SMOTE(train.data[, -1], train.data$LeagueIndex, K = 5, dup_size = 5)
train.data.smote <- train.smote$data
names(train.data.smote)[names(train.data.smote) == "class"] <- "LeagueIndex"
train.data.smote$LeagueIndex <- as.factor(train.data.smote$LeagueIndex)

train.data.smote.tomek <- tomek(train.data.smote, var = "LeagueIndex")

SMOTE is first applied to oversample the minority class by generating synthetic samples
between existing observations and their nearest neighbors. Tomek Links are then applied
to remove majority class samples that border minority class regions, cleaning the
decision boundary. Both techniques are applied only to the training set to prevent
data leakage.

In [ ]:
before.dist <- table(train.data$LeagueIndex)
before.pct <- round(before.dist / sum(before.dist) * 100, 2)

after.dist <- table(train.data.smote.tomek$LeagueIndex)
after.pct <- round(after.dist / sum(after.dist) * 100, 2)

resampling.summary <- data.frame(
  League = names(before.dist),
  Before_Count = as.numeric(before.dist),
  Before_Pct = paste0(before.pct, "%"),
  After_Count = as.numeric(after.dist),
  After_Pct = paste0(after.pct, "%"),
  row.names = NULL
)

print(resampling.summary, row.names = FALSE)

## 6.3 Machine Learning Models

### Logistic Regression

Features: 14 selected via backward elimination (Age, APM, MinimapRightClicks, and
ComplexAbilitiesUsed removed). The model is trained using maximum likelihood estimation
with a maximum of 500 iterations. Decay parameter is tuned via 5-fold cross-validation,
with decay = 0 selected as optimal.

In [ ]:
train_lr <- function(train.data) {
  set.seed(42)
  multinom(LeagueIndex ~ HoursPerWeek + TotalHours +
             SelectByHotkeys + AssignToHotkeys + UniqueHotkeys +
             MinimapAttacks + NumberOfPACs + GapBetweenPACs +
             ActionLatency + ActionsInPAC + TotalMapExplored +
             WorkersMade + UniqueUnitsMade + ComplexUnitsMade,
           data = train.data, maxit = 500, trace = FALSE)
}

train_lr_weighted <- function(data) {
  set.seed(42)
  weights <- calculate_class_weights(data)
  multinom(LeagueIndex ~ HoursPerWeek + TotalHours +
             SelectByHotkeys + AssignToHotkeys + UniqueHotkeys +
             MinimapAttacks + NumberOfPACs + GapBetweenPACs +
             ActionLatency + ActionsInPAC + TotalMapExplored +
             WorkersMade + UniqueUnitsMade + ComplexUnitsMade,
           data = data, weights = weights, maxit = 500, trace = FALSE)
}

### Decision Tree

Decision Trees use CART to recursively partition the feature space. All available
features are included, with pruning applied based on cross-validation error (xerror)
to prevent overfitting.

In [ ]:
train_dt <- function(data) {
  set.seed(42)
  model   <- rpart(LeagueIndex ~ ., data = data, method = "class")
  best.cp <- model$cptable[which.min(model$cptable[, "xerror"]), "CP"]
  prune(model, cp = best.cp)
}

train_dt_weighted <- function(data) {
  set.seed(42)
  weights <- calculate_class_weights(data)
  model   <- rpart(LeagueIndex ~ ., data = data, method = "class", weights = weights)
  best.cp <- model$cptable[which.min(model$cptable[, "xerror"]), "CP"]
  prune(model, cp = best.cp)
}

### Random Forest

Random Forest builds an ensemble of 1,000 decision trees using bootstrap aggregating
(bagging) with random feature selection at each split. A proximity matrix is computed
to enable distance-based diagnostics. All available features are included, consistent
with the approach used by Thompson et al. (2013).

In [ ]:
train_rf <- function(data) {
  set.seed(42)
  randomForest(LeagueIndex ~ ., data = data, ntree = 1000, proximity = TRUE)
}

train_rf_weighted <- function(data) {
  set.seed(42)
  weights <- calculate_class_weights(data)
  randomForest(LeagueIndex ~ ., data = data, ntree = 1000, proximity = TRUE,
               classwt = table(data$LeagueIndex) / nrow(data))
}

## 6.4 Model Evaluation

A unified evaluation function is applied across all three models. It generates a
confusion matrix on the test set and reports two metrics: overall accuracy and
recall for the minority class.

In [ ]:
eval_model <- function(model, test.data) {
  pred <- predict(model, newdata = test.data, type = "class")
  cm   <- confusionMatrix(pred, test.data$LeagueIndex)
  print(cm$table)
  min_class <- names(which.min(table(test.data$LeagueIndex)))
  results <- data.frame(
    Accuracy = round(cm$overall["Accuracy"], 4),
    Recall_minority = round(cm$byClass[paste0("Class: ", min_class), "Recall"], 4)
  )
  cat("Minority class: League", min_class, "\n")
  print(results, row.names = FALSE)
  invisible(results)
}

# 7. Results

## 7.1 Logistic Regression Results

### Baseline

In [ ]:
lr.model <- train_lr(train.data)
lr.results <- eval_model(lr.model, test.data)

The baseline model achieves an accuracy of 0.4145, but performs poorly on League 1
(Bronze), with a minority class recall of only 0.2727. The confusion matrix shows
that most Bronze players are misclassified as Silver or Gold.

### Class Weights

In [ ]:
lr.model.weighted <- train_lr_weighted(train.data)
lr.results.weighted <- eval_model(lr.model.weighted, test.data)

Class Weights improves minority recall substantially from 0.2727 to 0.5152, at the
cost of overall accuracy dropping from 0.4145 to 0.3812. The confusion matrix shows
League 1 (Bronze) is now correctly identified more often, though misclassification
across mid-tier leagues increases.

### SMOTE + Tomek

In [ ]:
lr.model.smote.tomek <- train_lr(train.data.smote.tomek)
lr.results.smote.tomek <- eval_model(lr.model.smote.tomek, test.data)

SMOTE + Tomek achieves the highest minority recall of 0.9091, correctly identifying
the majority of Bronze players. However, overall accuracy drops to 0.3661, the lowest
among the three strategies. The confusion matrix shows severe misclassification in
League 2 and 3, suggesting the synthetic oversampling has disrupted the decision
boundaries for mid-tier leagues.

### Comparison

In [ ]:
comparison.lr <- data.frame(
  Method = c("Baseline", "Class Weights", "SMOTE + Tomek"),
  Accuracy = c(lr.results$Accuracy, 
               lr.results.weighted$Accuracy, 
               lr.results.smote.tomek$Accuracy),
  Recall_minority = c(lr.results$Recall_minority, 
                      lr.results.weighted$Recall_minority, 
                      lr.results.smote.tomek$Recall_minority)
)
print(comparison.lr, row.names = FALSE)

The baseline achieves an accuracy of 0.4145 but a minority recall of only 0.2727, indicating
most Bronze players are misclassified as Silver or Gold. Class Weights improves minority
recall to 0.5152 with only a marginal accuracy drop to 0.3812. SMOTE + Tomek achieves the
highest minority recall of 0.9091 but the lowest accuracy (0.3661), as synthetic oversampling
disrupts decision boundaries for mid-tier leagues. Class Weights achieves the best tradeoff for
this model.

## 7.2 Decision Tree Results

### Baseline

In [ ]:
dt.model <- train_dt(train.data)
dt.results <- eval_model(dt.model, test.data)

In [ ]:
rpart.plot(dt.model)

The baseline Decision Tree achieves an accuracy of 0.3767 with a minority recall of
0.4545. As shown in the tree diagram, ActionLatency is the primary splitting feature,
with the entire tree built around three thresholds of this variable. Notably, League 2
(Silver) appears as unused in the tree — the model predicts zero samples as Silver,
suggesting the pruned tree does not find a reliable split for this class. Performance
is strongest in the middle leagues (3–5).

### Class Weights

In [ ]:
dt.model.weighted <- train_dt_weighted(train.data)
dt.results.weighted <- eval_model(dt.model.weighted, test.data)

Class Weights results in a significant accuracy drop to 0.2874. The confusion matrix
reveals that the model collapses predictions for League 3 and 4 entirely to zero,
while League 2 absorbs a large portion of misclassified samples. This suggests that
class weighting disrupts the tree's splitting criteria, causing it to lose the ability
to distinguish mid-tier leagues.

### SMOTE + Tomek

In [ ]:
dt.model.smote.tomek <- train_dt(train.data.smote.tomek)
dt.results.smote.tomek <- eval_model(dt.model.smote.tomek, test.data)

SMOTE + Tomek achieves the highest minority recall of 0.9091, correctly identifying
most Bronze players. However, overall accuracy drops to 0.2844, the lowest across all
models. The confusion matrix shows that League 2 and 3 predictions collapse entirely
to zero, with the majority of samples absorbed into League 1 and League 5. This
suggests that the synthetic oversampling severely distorts the decision boundaries for
mid-tier leagues in tree-based models.

### Comparison

In [ ]:
comparison.dt <- data.frame(
  Method = c("Baseline", "Class Weights", "SMOTE + Tomek"),
  Accuracy = c(dt.results$Accuracy, 
               dt.results.weighted$Accuracy, 
               dt.results.smote.tomek$Accuracy),
  Recall_minority = c(dt.results$Recall_minority, 
                      dt.results.weighted$Recall_minority, 
                      dt.results.smote.tomek$Recall_minority)
)
print(comparison.dt, row.names = FALSE)

Unlike Logistic Regression, Class Weights does not improve minority recall for
Decision Trees — recall actually drops from 0.4545 to 0.3939, while accuracy falls
substantially to 0.2874. SMOTE + Tomek achieves the highest minority recall (0.9091)
but at the largest accuracy cost (0.2844). Decision Trees appear more sensitive to
imbalance strategy than Logistic Regression, with both strategies causing mid-tier
league predictions to collapse entirely.

## 7.3 Random Forest Results

### Baseline

In [ ]:
rf.model <- train_rf(train.data)
rf.results   <- eval_model(rf.model, test.data)

The baseline Random Forest achieves an accuracy of 0.4024 with a minority recall of
0.3333. Compared to the Logistic Regression baseline, accuracy is slightly lower but
the confusion matrix shows more balanced predictions across leagues, with no class
collapsing to zero. Performance is strongest in League 6 (Masters), where 75 out of
134 samples are correctly classified.

In [ ]:
oob.error.data <- data.frame(
    Trees = rep(1:nrow(rf.model$err.rate), times = 7), 
    Type  = rep(c("OOB", "1", "2", "3", "4", "5", "6"), 
                each = nrow(rf.model$err.rate)),
    Error = c(rf.model$err.rate[, "OOB"],
              rf.model$err.rate[, "1"],
              rf.model$err.rate[, "2"],
              rf.model$err.rate[, "3"],
              rf.model$err.rate[, "4"],
              rf.model$err.rate[, "5"],
              rf.model$err.rate[, "6"])
)

ggplot(data = oob.error.data, aes(x = Trees, y = Error)) +
    geom_line(aes(color = Type)) +
    labs(title = "OOB Error Convergence (Baseline RF)",
         x = "Number of Trees", y = "Error Rate")

The OOB error stabilizes by approximately 300-400 trees, with most per-class errors
converging by that point. League 6 converges earliest (~100 trees), while League 3
shows the most variability throughout. ntree=1000 ensures all classes are
well-converged with no benefit from additional trees.

### Class Weights

In [ ]:
rf.model.weighted <- train_rf_weighted(train.data)
rf.results.weighted <- eval_model(rf.model.weighted, test.data)

Random Forest with Class Weights achieves an accuracy of 0.4070 with a minority
recall of 0.3333, virtually identical to the baseline. Unlike Logistic Regression
and Decision Tree, class weighting has minimal impact on Random Forest performance,
suggesting the ensemble method is already relatively robust to class imbalance
through its bagging mechanism.

### SMOTE + Tomek

In [ ]:
rf.model.smote.tomek <- train_rf(train.data.smote.tomek)
rf.results.smote.tomek <- eval_model(rf.model.smote.tomek, test.data)

SMOTE + Tomek improves minority recall to 0.6667, a notable increase from 0.3333,
while accuracy drops to 0.3933. Compared to Logistic Regression and Decision Tree,
the accuracy cost is smaller, suggesting Random Forest is more resilient to the
distortion introduced by synthetic oversampling.

### Diagnostic Plots

In [ ]:
varImpPlot(rf.model.smote.tomek)

The variable importance plot confirms that ActionLatency and APM are the two most
influential predictors, consistent with Thompson et al.'s finding that action speed
dominates skill prediction. NumberOfPACs and TotalHours follow as the next most
important features. Demographic features such as Age and HoursPerWeek rank in the
lower half, suggesting that in-game behavioral metrics are more predictive of skill
than time investment alone. ComplexUnitsMade is the least important feature overall.

In [ ]:
distance.matrix <- dist(1 - rf.model.smote.tomek$proximity)
mds.stuff <- cmdscale(distance.matrix, eig = TRUE, x.ret = TRUE)
mds.var.per <- round(mds.stuff$eig/sum(mds.stuff$eig)*100, 1)

mds.values <- mds.stuff$points

mds.data <- data.frame(Sample = rownames(mds.values),
    X = mds.values[, 1],
    Y = mds.values[, 2],
    Status = train.data.smote.tomek$LeagueIndex)

ggplot(data = mds.data, aes(x = X, y = Y, label = Sample)) +
    geom_text(aes(color = Status)) + 
    theme_bw() + 
    xlab(paste("MDS1 - ", mds.var.per[1], "%", sep = "")) +
    ylab(paste("MDS2 - ", mds.var.per[2], "%", sep = "")) +
    ggtitle("MDS plot using (1 - Random Forest Proximities)")

The MDS plot visualizes the proximity structure of the SMOTE + Tomek training data
based on Random Forest similarities. MDS1 explains 42.1% of the variance, clearly
separating League 1 (red) on the left from higher leagues on the right. Leagues 5
and 6 (blue and purple) form a tight cluster on the right, indicating they are most
similar to each other. Mid-tier leagues (2–4) overlap substantially in the center,
consistent with the difficulty in distinguishing them observed in the confusion matrices.

### Comparison

In [ ]:
comparison.rf <- data.frame(
  Method = c("Baseline", "Class Weights", "SMOTE + Tomek"),
  Accuracy = c(rf.results$Accuracy, 
               rf.results.weighted$Accuracy, 
               rf.results.smote.tomek$Accuracy),
  Recall_minority = c(rf.results$Recall_minority, 
                      rf.results.weighted$Recall_minority, 
                      rf.results.smote.tomek$Recall_minority)
)
print(comparison.rf, row.names = FALSE)

Random Forest is the most stable model across all three imbalance strategies. Class
Weights achieves a marginal accuracy improvement to 0.4070 with no change in minority
recall, while SMOTE + Tomek improves recall to 0.6667 with only a modest accuracy drop
to 0.3933. Unlike Decision Tree, no class collapses to zero under any strategy.

## 7.4 Pairwise Binary Classification

In [ ]:
run_pairwise_rf <- function(league_a, league_b) {
  
  pair.data <- skillcraft.data.imputed[
    skillcraft.data.imputed$LeagueIndex %in% c(league_a, league_b), ]
  pair.data$LeagueIndex <- droplevels(pair.data$LeagueIndex)
  
  set.seed(42)
  idx        <- sample(1:nrow(pair.data), 0.8 * nrow(pair.data))
  pair.train <- pair.data[idx, ]
  pair.test  <- pair.data[-idx, ]
  
  model <- train_rf_weighted(pair.train)
  pred  <- predict(model, newdata = pair.test)
  acc   <- round(mean(pred == pair.test$LeagueIndex) * 100, 2)
  
  return(acc)
}

In [ ]:
acc.bg <- run_pairwise_rf(1, 3)
acc.sp <- run_pairwise_rf(2, 4)
acc.gd <- run_pairwise_rf(3, 5)
acc.pm <- run_pairwise_rf(4, 6)

paper.acc <- c(82.32, 78.27, 79.01, 80.70)
our.acc   <- c(acc.bg, acc.sp, acc.gd, acc.pm)

pairwise.summary <- data.frame(
  Pair         = c("Bronze-Gold", "Silver-Platinum", "Gold-Diamond", "Platinum-Masters"),
  Paper_Forest = paper.acc,
  Our_LR       = our.acc,
  Beat_Paper   = our.acc > paper.acc
)
print(pairwise.summary, row.names = FALSE)

To enable direct comparison with Thompson et al. (2013), we replicated their pairwise
binary classification setup using our best-performing configuration (RF + Class Weights).

Our model outperforms their Conditional Inference Forest results in 3 out of 4 pairs.
The only exception is Gold-Diamond (League 3 vs 5), where our model scores 75.00%
compared to their 79.01%. This pair likely reflects the inherent difficulty of
distinguishing mid-tier leagues, where behavioral differences between players are
less pronounced.

Detailed discussion of these results in relation to Thompson et al. is provided in
Section 8.3.

## 7.5 Overall Model Comparison

In [ ]:
all.results <- data.frame(
  Algorithm = rep(c("Logistic Regression", "Decision Tree", "Random Forest"), each = 3),
  Method    = rep(c("Baseline", "Class Weights", "SMOTE + Tomek"), 3),
  Accuracy  = c(comparison.lr$Accuracy,
                comparison.dt$Accuracy,
                comparison.rf$Accuracy),
  Recall_minority = c(comparison.lr$Recall_minority,
                      comparison.dt$Recall_minority,
                      comparison.rf$Recall_minority)
)

print(all.results, row.names = FALSE)

The table above summarizes the performance of all nine model-strategy combinations.

- RF + Class Weights achieves the best balance, with the highest accuracy among
  imbalance-aware strategies (0.4070)
- SMOTE + Tomek consistently produces the highest minority recall across all three
  models, but at a significant cost to overall accuracy
- Decision Tree is the weakest performer overall, with accuracy dropping below 0.30
  under both Class Weights and SMOTE + Tomek
- Logistic Regression Baseline achieves the highest raw accuracy (0.4145), but its
  minority recall of 0.2727 indicates it largely ignores the minority class
- Given the class imbalance objective of this project, RF + Class Weights is selected
  as the best overall model

# 8. Discussion

## 8.1 Imbalance Strategy Comparison

**Baseline**

Pros:
- Preserves original data distribution without modification
- Achieves the highest raw accuracy among all nine configurations

Cons:
- Largely ignores minority class, resulting in poor recall for League 1

---

**Class Weights**

Pros:
- No changes to training data, all original information is preserved
- Meaningful minority recall improvement with only marginal accuracy cost

Cons:
- Does not address boundary overlap between adjacent leagues

---

**SMOTE + Tomek**

Pros:
- Produces the highest minority recall across all models
- Tomek links remove borderline samples, potentially reducing boundary overlap

Cons:
- Synthetic samples may not reflect real player behavior patterns
- Tomek removal reduces majority class samples, discarding useful training information
- Largest overall accuracy drop among the three strategies

---

Among the three strategies, Class Weights offers the best tradeoff between accuracy and
minority recall, and is therefore selected as the preferred strategy in this project.

## 8.2 Multinomial vs. Pairwise Classification

Pairwise binary classification simplifies the problem by isolating two leagues at a time,
naturally yielding higher accuracy. Multinomial classification — predicting all six leagues
simultaneously — is a fundamentally harder task, as adjacent leagues share similar
behavioral profiles and the model must learn to distinguish all boundaries at once.

Interpreted in the context of a 6-class problem where random chance yields only 16.7%,
the accuracy range of 0.28–0.42 is reasonable. Class Weights provides the most meaningful
improvement, balancing minority recall gains without sacrificing overall discriminative power.

## 8.3 Comparison with Thompson et al.

Thompson et al. (2013) applied Conditional Inference Forest to pairwise binary
classification across adjacent leagues, achieving accuracies ranging from 78.27%
to 82.32%. Using RF + Class Weights, we replicated this setup and outperformed
their results in 3 out of 4 pairs.

The only exception is Gold-Diamond (League 3 vs 5), where our model scores 75.00%
compared to their 79.01%. Mid-tier leagues share the most similar behavioral profiles,
making this boundary the hardest to separate regardless of model configuration.

It should also be noted that direct comparison has limitations. Their study framed
the problem as a series of binary decisions, while this study addresses 6-class
multinomial classification — a fundamentally harder task. The pairwise results here
serve as a reference point rather than a strict benchmark.

## 8.4 Limitations & Future Work

Limitations

- Leagues 7 and 8 were excluded due to data quality issues, limiting the scope of analysis
- The relatively small training set constrains model performance, particularly for minority classes

Future Work

- Ordinal Logistic Regression — League ranks are inherently ordered (1 to 6);
  an ordinal model would better capture this structure compared to standard multinomial
  classification
- XGBoost / Gradient Boosting — known for strong performance on imbalanced tabular
  data, and may handle league boundary overlap more effectively than Decision Trees
- Longitudinal analysis — tracking the same players over time could reveal how
  behavioral patterns evolve with skill development

# . References

Thompson, J. J., Blair, M. R., Chen, L., & Henrey, A. J. (2013). Video game telemetry
as a critical tool in the study of complex skill learning. *PLOS ONE, 8*(9), e75129.
https://doi.org/10.1371/journal.pone.0075129

Blair, M., Thompson, J., Henrey, A., & Chen, B. (2013). SkillCraft1 Master Table Dataset
[Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5161N

# 10. Appendix

The complete R code for this analysis is available at:

https://github.com/ShengPeiWilliam/skillcraft-ml